# Regression

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
from os.path import join
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, cross_validate, cross_val_score, KFold
from sklearn.ensemble import HistGradientBoostingRegressor
from config import ROOT_DIR

base_data_folderpath = join(ROOT_DIR, "datasets/multi_news/similarities/")
epsilons = [1] + [i for i in range(5, 101, 5)]

np.random.seed(42) # Reproductibility

## Loading data

In [3]:
ogtext_gensum_sim_filepath = "OGtextVSbartGensummarySimilarities.pickle"

with open(os.path.join(base_data_folderpath, ogtext_gensum_sim_filepath), "rb") as f:
    ogtext_gensum_sim = np.array(pickle.load(f))

ogtext_noisygensum_sim_files = {
    1:"OGtextVSbartnoisygenSummaryEpsi1Similarities.pickle",
    5:"OGtextVSbartnoisygenSummaryEpsi5Similarities.pickle",
    10:"OGtextVSbartnoisygenSummaryEpsi10Similarities.pickle",
    15:"OGtextVSbartnoisygenSummaryEpsi15Similarities.pickle",
    20:"OGtextVSbartnoisygenSummaryEpsi20Similarities.pickle",
    25:"OGtextVSbartnoisygenSummaryEpsi25Similarities.pickle",
    30:"OGtextVSbartnoisygenSummaryEpsi30Similarities.pickle",
    35:"OGtextVSbartnoisygenSummaryEpsi35Similarities.pickle",
    40:"OGtextVSbartnoisygenSummaryEpsi40Similarities.pickle",
    45:"OGtextVSbartnoisygenSummaryEpsi45Similarities.pickle",
    50:"OGtextVSbartnoisygenSummaryEpsi50Similarities.pickle",
    55:"OGtextVSbartnoisygenSummaryEpsi55Similarities.pickle",
    60:"OGtextVSbartnoisygenSummaryEpsi60Similarities.pickle",
    65:"OGtextVSbartnoisygenSummaryEpsi65Similarities.pickle",
    70:"OGtextVSbartnoisygenSummaryEpsi70Similarities.pickle",
    75:"OGtextVSbartnoisygenSummaryEpsi75Similarities.pickle",
    80:"OGtextVSbartnoisygenSummaryEpsi80Similarities.pickle",
    85:"OGtextVSbartnoisygenSummaryEpsi85Similarities.pickle",
    90:"OGtextVSbartnoisygenSummaryEpsi90Similarities.pickle",
    95:"OGtextVSbartnoisygenSummaryEpsi95Similarities.pickle",
    100:"OGtextVSbartnoisygenSummaryEpsi100Similarities.pickle"
}
ogtext_noisygensum_sim = {}
for i in sorted(ogtext_noisygensum_sim_files.keys()):
    with open(os.path.join(base_data_folderpath, ogtext_noisygensum_sim_files[i]), "rb") as f:
        ogtext_noisygensum_sim[i] = pickle.load(f)

In [5]:
ogtext_llamagensum_sim_file = "OGtextVSLlamaGensummarySimilarities.pickle"

with open(os.path.join(base_data_folderpath, ogtext_llamagensum_sim_file), "rb") as f:
    ogtext_llamagensum_sim = np.array(pickle.load(f))

ogtext_llamanoisygensum_sim_files = {
    1:"OGtextVSLlamanoisygenSummaryEpsi1Similarities.pickle",
    5:"OGtextVSLlamanoisygenSummaryEpsi5Similarities.pickle",
    10:"OGtextVSLlamanoisygenSummaryEpsi10Similarities.pickle",
    15:"OGtextVSLlamanoisygenSummaryEpsi15Similarities.pickle",
    20:"OGtextVSLlamanoisygenSummaryEpsi20Similarities.pickle",
    25:"OGtextVSLlamanoisygenSummaryEpsi25Similarities.pickle",
    30:"OGtextVSLlamanoisygenSummaryEpsi30Similarities.pickle",
    35:"OGtextVSLlamanoisygenSummaryEpsi35Similarities.pickle",
    40:"OGtextVSLlamanoisygenSummaryEpsi40Similarities.pickle",
    45:"OGtextVSLlamanoisygenSummaryEpsi45Similarities.pickle",
    50:"OGtextVSLlamanoisygenSummaryEpsi50Similarities.pickle",
    55:"OGtextVSLlamanoisygenSummaryEpsi55Similarities.pickle",
    60:"OGtextVSLlamanoisygenSummaryEpsi60Similarities.pickle",
    65:"OGtextVSLlamanoisygenSummaryEpsi65Similarities.pickle",
    70:"OGtextVSLlamanoisygenSummaryEpsi70Similarities.pickle",
    75:"OGtextVSLlamanoisygenSummaryEpsi75Similarities.pickle",
    80:"OGtextVSLlamanoisygenSummaryEpsi80Similarities.pickle",
    85:"OGtextVSLlamanoisygenSummaryEpsi85Similarities.pickle",
    90:"OGtextVSLlamanoisygenSummaryEpsi90Similarities.pickle",
    95:"OGtextVSLlamanoisygenSummaryEpsi95Similarities.pickle",
    100:"OGtextVSLlamanoisygenSummaryEpsi100Similarities.pickle",
}

ogtext_llamanoisygensum_sim = {}
for i in sorted(ogtext_llamanoisygensum_sim_files.keys()):
    with open(os.path.join(base_data_folderpath, ogtext_llamanoisygensum_sim_files[i]), "rb") as f:
        ogtext_llamanoisygensum_sim[i] = pickle.load(f)

In [6]:
nb_texts = ogtext_gensum_sim.shape[0]
nb_epsilons = len(epsilons)

## Perform Regression

In [7]:
# The features are: epsilon, ogtext_gensum_sim, ogtext_noisygensum_sim
# The target is: ogtext_llamanoisygensum_sim

df2 = pd.DataFrame(
    {
        "epsilon":np.repeat(epsilons, nb_texts),
        "ogtext_gensum_sim":np.tile(ogtext_gensum_sim, nb_epsilons),
        "ogtext_noisygensum_sim": np.array(list(ogtext_noisygensum_sim.values())).flatten(),
        "ogtext_llamanoisygensum_sim": np.array(list(ogtext_llamanoisygensum_sim.values())).flatten()
    }
)

target = df2["ogtext_llamanoisygensum_sim"]
data = df2.drop(columns=["ogtext_llamanoisygensum_sim"])

### With Pipeline and cross_val_score
to get a general idea of the performances

In [ ]:
model = make_pipeline(StandardScaler(), HistGradientBoostingRegressor())

# Define the cross-validation strategy
cv = KFold(n_splits=5, shuffle=True)

# Define the scoring metrics
scoring = ["r2", "neg_root_mean_squared_error"]

# Perform cross-validation
cv_result = cross_validate(model, data, target, cv=cv, scoring=scoring)

# Print the results for both metrics
print("R² scores:", cv_result['test_r2'])
print("Mean R² score:", np.mean(cv_result['test_r2']))

print("RMSE scores:", -cv_result['test_neg_root_mean_squared_error'])
print("Mean Negative RMSE score:", np.mean(-cv_result['test_neg_root_mean_squared_error']))
